## Deep Research Agent

This notebook builds a multi-agent deep research pipeline with a Gradio UI. Here is what each cell does:

1. **Cell 1 — Imports & Setup**: Load environment variables, import all dependencies.
2. **Cell 2 — Schemas & Guardrails**: Define Pydantic structured output models and input guardrails to block off-topic or harmful queries.
3. **Cell 3 — Agent Definitions**: Define the Planner, Search, Writer, Subject Writer, and Email agents with model settings.
4. **Cell 4 — Pipeline Functions**: Async functions that wire all agents together into a research pipeline.
5. **Cell 5 — Email Tool**: The SendGrid function tool used by the email agent to dispatch the final report.
6. **Cell 6 — Gradio UI**: Launch an interactive UI to run the full pipeline from a browser.

### Cell 1 — Imports & Setup

Load all required libraries and environment variables. We use `openai-agents` for the agentic framework, `sendgrid` for email dispatch, and `gradio` for the UI.

In [1]:
import asyncio
import os
from typing import Dict

import gradio as gr
import sendgrid
from agents import (
    Agent,
    GuardrailFunctionOutput,
    Runner,
    WebSearchTool,
    function_tool,
    gen_trace_id,
    input_guardrail,
    trace,
)
from agents.model_settings import ModelSettings
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from sendgrid.helpers.mail import Content, Email, Mail, To

load_dotenv(override=True)
print("Environment loaded.")

Environment loaded.


### Cell 2 — Schemas & Guardrails

We define:
- **Pydantic models** for structured outputs from the Planner and Writer agents, ensuring well-typed, predictable responses.
- **An input guardrail** that uses a fast model to screen queries before the expensive pipeline runs — blocking harmful, nonsensical, or off-topic input.

In [2]:
# ── Structured Output Schemas ──────────────────────────────────────────────────

class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search is important to the query.")
    query: str = Field(description="The search term to use.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="List of web searches to perform.")

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of findings.")
    markdown_report: str = Field(description="The full detailed report in markdown.")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further.")

class GuardrailOutput(BaseModel):
    is_valid: bool = Field(description="True if the query is a legitimate research topic.")
    reason: str = Field(description="Why the query was accepted or rejected.")


# ── Guardrail Agent ────────────────────────────────────────────────────────────

guardrail_agent = Agent(
    name="GuardrailAgent",
    instructions=(
        "Evaluate whether a user's research query is legitimate, safe, and meaningful. "
        "Reject queries that are harmful, nonsensical, contain personal data requests, "
        "or are clearly not research topics. Return is_valid=True for genuine research queries."
    ),
    model="gpt-4o-mini",
    output_type=GuardrailOutput,
)


@input_guardrail
async def research_guardrail(ctx, agent, input_data):
    result = await Runner.run(guardrail_agent, input_data, context=ctx.context)
    output = result.final_output
    return GuardrailFunctionOutput(
        output_info=output,
        tripwire_triggered=not output.is_valid,
    )

print("Schemas and guardrails defined.")

Schemas and guardrails defined.


### Cell 3 — Agent Definitions

We define five agents, each with a focused responsibility:
- **Planner**: Breaks the query into targeted web searches.
- **Search**: Executes each search using the `WebSearchTool` and returns a concise summary.
- **Writer**: Synthesises all search summaries into a detailed markdown report.
- **Subject Writer**: Generates a compelling email subject line from the report.
- **Email Agent**: Converts the report to HTML and sends it via the email tool.

`ModelSettings` is used to control temperature and tool choice per agent.

In [3]:
HOW_MANY_SEARCHES = 3

# ── Planner Agent ──────────────────────────────────────────────────────────────
planner_agent = Agent(
    name="PlannerAgent",
    instructions=(
        f"You are a research planner. Given a query, output exactly {HOW_MANY_SEARCHES} "
        "targeted web search terms that together would best answer the query comprehensively."
    ),
    model="gpt-4o-mini",
    model_settings=ModelSettings(temperature=0.3),
    output_type=WebSearchPlan,
    input_guardrails=[research_guardrail],
)

# ── Search Agent ───────────────────────────────────────────────────────────────
search_agent = Agent(
    name="SearchAgent",
    instructions=(
        "You are a research assistant. Search the web for the given term and produce a "
        "concise 2-3 paragraph summary under 300 words. Capture key facts only. "
        "No commentary, no filler — just the essence of the findings."
    ),
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required", temperature=0.2),
)

# ── Writer Agent ───────────────────────────────────────────────────────────────
writer_agent = Agent(
    name="WriterAgent",
    instructions=(
        "You are a senior researcher. Given a query and summarised search results, "
        "write a cohesive, detailed markdown report of at least 1000 words (5-10 pages). "
        "Start with an outline, then produce the full report with headings and structure."
    ),
    model="gpt-4o-mini",
    model_settings=ModelSettings(temperature=0.4),
    output_type=ReportData,
)

# ── Subject Writer Agent ───────────────────────────────────────────────────────
subject_agent = Agent(
    name="SubjectAgent",
    instructions=(
        "Given a research report, generate a concise, compelling email subject line "
        "of no more than 10 words. Return ONLY the subject line text, nothing else."
    ),
    model="gpt-4o-mini",
    model_settings=ModelSettings(temperature=0.5),
)

# ── Email Agent ────────────────────────────────────────────────────────────────
email_agent = Agent(
    name="EmailAgent",
    instructions=(
        "You send research reports by email. Convert the provided markdown report into "
        "clean, well-structured HTML. Then call the send_email tool with the subject and HTML body."
    ),
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required", temperature=0.2),
)

print("All agents defined.")

All agents defined.


### Cell 4 — Email Tool & Pipeline Functions

We define the `send_email` function tool (used by the Email Agent to call SendGrid) and the async pipeline functions that chain all agents together:
- `plan_searches` → `perform_searches` → `write_report` → `get_subject` → `dispatch_email`

The top-level `run_pipeline` function orchestrates everything and streams progress messages for the Gradio UI.

In [4]:
# ── Email Tool ─────────────────────────────────────────────────────────────────

@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send an email with the given subject and HTML body via SendGrid."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email(os.environ.get("SENDGRID_FROM_EMAIL"))  # your verified sender
    to_email = To(os.environ.get("SENDGRID_TO_EMAIL"))          # recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "sent", "subject": subject}

# Attach tool to email agent after definition
email_agent.tools = [send_email]


# ── Pipeline Functions ─────────────────────────────────────────────────────────

async def plan_searches(query: str) -> WebSearchPlan:
    result = await Runner.run(planner_agent, f"Query: {query}")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan) -> list[str]:
    tasks = [
        asyncio.create_task(
            Runner.run(search_agent, f"Search term: {item.query}\nReason: {item.reason}")
        )
        for item in search_plan.searches
    ]
    results = await asyncio.gather(*tasks)
    return [r.final_output for r in results]

async def write_report(query: str, search_results: list[str]) -> ReportData:
    input_text = f"Original query: {query}\nSearch summaries:\n" + "\n---\n".join(search_results)
    result = await Runner.run(writer_agent, input_text)
    return result.final_output

async def get_subject(report: ReportData) -> str:
    result = await Runner.run(subject_agent, report.markdown_report[:1000])
    return result.final_output.strip()

async def dispatch_email(report: ReportData, subject: str):
    prompt = f"Subject: {subject}\n\nReport:\n{report.markdown_report}"
    await Runner.run(email_agent, prompt)


# ── Top-Level Orchestrator ─────────────────────────────────────────────────────

async def run_pipeline(query: str):
    """Run the full research pipeline and yield progress updates."""
    trace_id = gen_trace_id()
    log = []

    def update(msg):
        log.append(msg)
        return "\n".join(log)

    with trace("Deep Research", trace_id=trace_id):
        yield update(f"🔍 Planning searches for: *{query}*")
        try:
            search_plan = await plan_searches(query)
        except Exception as e:
            yield update(f"❌ Guardrail blocked query: {e}")
            return

        yield update(f"📋 Will run {len(search_plan.searches)} searches...")
        search_results = await perform_searches(search_plan)
        yield update("✅ Searches complete.")

        yield update("✍️ Writing report...")
        report = await write_report(query, search_results)
        yield update("📄 Report complete.")

        yield update("🏷️ Generating subject line...")
        subject = await get_subject(report)
        yield update(f"📧 Subject: *{subject}*")

        yield update("📨 Sending email...")
        await dispatch_email(report, subject)
        yield update("✅ Email sent!")

        yield update(f"\n---\n**Summary:** {report.short_summary}")
        yield update(f"\n**Follow-up questions:**\n" + "\n".join(f"- {q}" for q in report.follow_up_questions))
        yield update(f"\n---\n**Full Report:**\n\n{report.markdown_report}")

print("Pipeline and email tool ready.")

Pipeline and email tool ready.


### Cell 5 — Add SendGrid Email Env Variables

Before running, make sure your `.env` file contains these three variables. The pipeline reads them at runtime:

```
OPENAI_API_KEY=sk-...
SENDGRID_API_KEY=SG....
SENDGRID_FROM_EMAIL=you@yourdomain.com   # must be verified in SendGrid
SENDGRID_TO_EMAIL=recipient@example.com
```

Run this cell to confirm the keys are loaded correctly.

In [5]:
required_keys = ["OPENAI_API_KEY", "SENDGRID_API_KEY", "SENDGRID_FROM_EMAIL", "SENDGRID_TO_EMAIL"]

for key in required_keys:
    val = os.environ.get(key)
    status = "✅ Set" if val else "❌ Missing"
    print(f"{key}: {status}")

OPENAI_API_KEY: ✅ Set
SENDGRID_API_KEY: ✅ Set
SENDGRID_FROM_EMAIL: ✅ Set
SENDGRID_TO_EMAIL: ✅ Set


### Cell 6 — Gradio UI

Launch an interactive Gradio interface. Enter any research topic, click **Run Research**, and watch the pipeline execute step by step. The final report is streamed into the output panel, and the formatted report is sent to your email automatically.

In [6]:
async def gradio_pipeline(query: str):
    """Wrapper to stream pipeline output into Gradio."""
    output = ""
    async for chunk in run_pipeline(query):
        output = chunk
        yield output


with gr.Blocks(title="Deep Research Agent") as demo:
    gr.Markdown("## 🔬 Deep Research Agent\nEnter a topic and the agent will research it, write a report, and email it to you.")

    with gr.Row():
        query_input = gr.Textbox(
            label="Research Topic",
            placeholder="e.g. Latest AI Agent frameworks in 2025",
            scale=4,
        )
        run_btn = gr.Button("Run Research", variant="primary", scale=1)

    output_box = gr.Markdown(label="Pipeline Output")

    run_btn.click(
        fn=gradio_pipeline,
        inputs=query_input,
        outputs=output_box,
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
